# Ablation: Weight Decay (WD)

Comparing different weight decay values:
- **WD=1e-3**: Lower weight decay
- **WD=5e-3**: Medium-low weight decay
- **WD=1e-2 (Default)**: Default weight decay (baseline)
- **WD=2e-2**: Medium-high weight decay
- **WD=5e-2**: Higher weight decay

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import numpy as np
from pathlib import Path

# Import shared ablation utilities
from ablation_utils import (
    setup_plotting_style,
    load_all_ablation_models,
    load_all_models_all_metrics,
    make_latex_ablation_table,
    plot_ablation_line,
    plot_ablation_bars,
    compute_deltas,
    print_summary,
    METRICS, METRIC_DISPLAY, METRIC_COLORS
)

# Set up plotting style
setup_plotting_style()

In [ ]:
# =============================================================================
# CONFIGURATION - Define ablation models
# =============================================================================

ABLATION_MODELS = {
    "WD=1e-3": {
        "csv_path": "../evaluation/ablations/01-Jan_coco_with_components_negatives_structured_rel1.0_either_max2_lf1.0_lc0.5_negclip_hard_lr5ee-6_wd1e-3_neg_rel0.2_inplace1.0_swap1.0_ablation_wd_1e-3.csv",
        "is_baseline": False,
        "description": "Lower weight decay",
        "wd_value": 1e-3
    },
    "WD=5e-3": {
        "csv_path": "../evaluation/ablations/01-Jan_coco_with_components_negatives_structured_rel1.0_either_max2_lf1.0_lc0.5_negclip_hard_lr5ee-6_wd5e-3_neg_rel0.2_inplace1.0_swap1.0_ablation_wd_5e-3.csv",
        "is_baseline": False,
        "description": "Medium-low weight decay",
        "wd_value": 5e-3
    },
    "CS-CLIP (WD=1e-2)": {
        "csv_path": "../evaluation/exp_csv/19-Dec_coco_with_components_negatives_structured_rel1.0_either_max2_lf1.0_lc0.5_negclip_hard_lr5ee-6_wd1e-2_neg_rel0.0_inplace1.0_swap1.0_csclip-negclip-hard-new_cleaned.csv",
        "is_baseline": True,
        "description": "Default weight decay (baseline)",
        "wd_value": 1e-2
    },
    "WD=2e-2": {
        "csv_path": "../evaluation/ablations/01-Jan_coco_with_components_negatives_structured_rel1.0_either_max2_lf1.0_lc0.5_negclip_hard_lr5ee-6_wd2e-2_neg_rel0.2_inplace1.0_swap1.0_ablation_wd_2e-2.csv",
        "is_baseline": False,
        "description": "Medium-high weight decay",
        "wd_value": 2e-2
    },
    "WD=5e-2": {
        "csv_path": "../evaluation/ablations/01-Jan_coco_with_components_negatives_structured_rel1.0_either_max2_lf1.0_lc0.5_negclip_hard_lr5ee-6_wd5e-2_neg_rel0.2_inplace1.0_swap1.0_ablation_wd_5e-2.csv",
        "is_baseline": False,
        "description": "Higher weight decay",
        "wd_value": 5e-2
    },
}

# Primary metric for comparison
PRIMARY_METRIC = "text_contrastive_accuracy"

# Checkpoint selection (use best or specific step)
CHECKPOINT_STEP = None  # None = use best checkpoint, or specify step like 5000

# Ablation metadata
ABLATION_NAME = "WEIGHT DECAY ABLATION"
PARAM_KEY = "wd_value"
PARAM_LABEL = 'Weight Decay'

print("Ablation: Weight Decay (WD)")
print("="*50)
for name, cfg in ABLATION_MODELS.items():
    baseline_mark = " [BASELINE]" if cfg["is_baseline"] else ""
    print(f"  {name}{baseline_mark}: {cfg['description']}")

In [ ]:
# =============================================================================
# LOAD DATA - Single Metric (Primary)
# =============================================================================

scores_df = load_all_ablation_models(ABLATION_MODELS, PRIMARY_METRIC, CHECKPOINT_STEP)
print(f"\nLoaded {len(scores_df)} models, {len(scores_df.columns)} datasets")

In [ ]:
# =============================================================================
# DISPLAY RAW SCORES TABLE
# =============================================================================

# Convert to percentage and display
scores_pct = scores_df * 100

# Add average column
scores_pct['Average'] = scores_pct.mean(axis=1)

print("\n" + "="*60)
print(f"ABLATION: WEIGHT DECAY")
print(f"Metric: {PRIMARY_METRIC}")
print("="*60)
display(scores_pct.round(1).style.highlight_max(axis=0, color='lightgreen'))

In [ ]:
# =============================================================================
# LOAD ALL METRICS (Text, Image, Group Contrastive Accuracy)
# =============================================================================

# Load all models with all metrics
all_metrics_df = load_all_models_all_metrics(ABLATION_MODELS, METRICS, CHECKPOINT_STEP)

# Extract just the summary columns (I2T, T2I, Group)
summary_cols = [col for col in ['I2T', 'T2I', 'Group'] if col in all_metrics_df.columns]
summary_df = all_metrics_df[summary_cols].copy()

# Add overall average
summary_df['Average'] = summary_df.mean(axis=1)

print("\n" + "="*60)
print("ABLATION: WEIGHT DECAY - ALL METRICS")
print("="*60)
display((summary_df * 100).round(1).style.highlight_max(axis=0, color='lightgreen'))

In [ ]:
# =============================================================================
# LATEX TABLE GENERATION
# =============================================================================

# Generate LaTeX table
latex_table = make_latex_ablation_table(
    summary_df,
    ABLATION_MODELS,
    caption="Weight decay ablation. I2T = Image-to-Text, T2I = Text-to-Image, Group = both correct. Best in \\textbf{bold}, baseline \\underline{underlined}.",
    label="tab:ablation_weight_decay",
)

print("="*60)
print("LATEX TABLE")
print("="*60)
print(latex_table)

In [ ]:
# =============================================================================
# VISUALIZATION: LINE PLOT (WD vs Performance)
# =============================================================================

# Extract WD values and corresponding scores
wd_values = [ABLATION_MODELS[model]['wd_value'] for model in summary_df.index]

fig, ax = plot_ablation_line(
    summary_df,
    wd_values,
    ABLATION_MODELS,
    param_label=PARAM_LABEL,
    title='Weight Decay Ablation',
    save_path='../paper_figures/ablation_weight_decay_line.pdf',
    log_scale=True  # Use log scale for weight decay
)

In [ ]:
# =============================================================================
# VISUALIZATION: GROUPED BAR CHART (All Metrics)
# =============================================================================

fig, ax = plot_ablation_bars(
    summary_df,
    ABLATION_MODELS,
    title='Weight Decay Ablation',
    save_path='../paper_figures/ablation_weight_decay_bars.pdf'
)

In [ ]:
# =============================================================================
# COMPUTE DELTAS FROM BASELINE
# =============================================================================

deltas_df = compute_deltas(summary_df, ABLATION_MODELS)

print("\n" + "="*60)
print("DELTA FROM BASELINE (percentage points)")
print("="*60)
display(deltas_df.round(2).style.background_gradient(cmap='RdYlGn', axis=None))

In [ ]:
# =============================================================================
# SUMMARY
# =============================================================================

print_summary(summary_df, ABLATION_MODELS, ABLATION_NAME, PARAM_KEY)

In [ ]:
# =============================================================================
# DATASET-WISE AND SUBSET-WISE TABLES (with ARO merging)
# =============================================================================

from ablation_utils import (
    load_all_models_per_dataset,
    load_all_models_per_subset,
    make_latex_dataset_table,
    get_datasets_and_subsets,
    display_all_tables,
    load_benchmark_config
)

# Load benchmark config for dataset merge rules (e.g., ARO)
bench_cfg = load_benchmark_config()

# Display all tables for the primary metric (I2T) with ARO merging
dataset_df, subset_df, datasets_subsets = display_all_tables(
    ABLATION_MODELS, PRIMARY_METRIC, CHECKPOINT_STEP, 
    show_latex=True, apply_merge=True, benchmark_config=bench_cfg
)